# Analysis of 23 years of climate data from Salamanca 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import os 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np 


from loaders import *
from build import *

In [ ]:
# build()

In [ ]:
#df = load_range(2000,1,2023,2)
#df.to_parquet("../data/station_data.parquet")


df = pd.read_parquet("../data/station_data.parquet")
df.head()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

print(df.index)  # espero un DateTime index 
df.head()

In [ ]:
fig, ax = plt.subplots(figsize = (10,5), dpi = 200)

montly_count = df.resample("ME").count()
expected = montly_count.index.days_in_month * 144

missing = montly_count.rsub(expected, axis = 0) # reverse substraction means expected-count

plt.grid()
plt.plot(missing.index, missing["temp"])
plt.xlabel("Year")
plt.ylabel("Number of missing values ")
plt.title("Missing values each year")

ax.text(pd.to_datetime("2003-01-01"),1700,"Before year 2001 a lot of \nsensors  weren't installed.")

plt.show()

In [ ]:
fig, ax  = plt.subplots(figsize=(10,5), dpi=200)

monthly = df["temp"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="red", label = "Temperature") # i established that the date is the index
plt.ylabel("ºC")
avg = monthly.values.mean()
plt.axhline(avg, color = "orange", linestyle = "--", label = f"Average Temperature {avg:.1f} ")
plt.legend()
plt.xlabel("Year")
plt.title("Temperature 2000-2023")

ax.xaxis.set_major_locator(mdates.YearLocator(1))
plt.xticks(rotation = 45)

plt.grid()
plt.show()

In [ ]:
fig, ax  = plt.subplots(figsize=(10,5), dpi=200)

monthly = df["humidity"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="blue", label = "Humidity") # i established that the date is the index
plt.ylabel("%")
avg = monthly.values.mean()
plt.axhline(avg, color = "steelblue", linestyle = "--", label = f"Average Humidity: {avg:.1f}%")
plt.legend()
plt.xlabel("Year")
plt.title("Humidity 2000-2023")

ax.xaxis.set_major_locator(mdates.YearLocator(1))
plt.xticks(rotation = 45)

plt.grid()
plt.show()

In [ ]:
fig, ax  = plt.subplots(figsize=(10,5), dpi=200)

monthly = df["vapor_pressure"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="dodgerblue", label = "Vapor Pressure") # i established that the date is the index
plt.ylabel("hPa")
avg = monthly.values.mean()
plt.axhline(avg, color = "slateblue", linestyle = "--", label = f"Average Vapor Pressure: {avg:.1f} hPa")
plt.legend()
plt.xlabel("Year")

plt.title("Vapor Pressure 2000-2023")

ax.xaxis.set_major_locator(mdates.YearLocator(1))
plt.xticks(rotation = 45)

plt.grid()
plt.show()

In [ ]:
import pandas as pd

# 1. Comprobar el tipo de datos de la columna original
print("Tipo de datos en la columna:", df["vapor_deficit"].dtype)

# 2. Ver si hay valores que no se pueden convertir a número (por ejemplo, textos o espacios vacíos)
no_numericos = df[pd.to_numeric(df["vapor_deficit"], errors="coerce").isna() & df["vapor_deficit"].notna()]
print(f"Número de valores no numéricos/extraños encontrados: {len(no_numericos)}")
if len(no_numericos) > 0:
    print("Ejemplos de valores extraños:")
    print(no_numericos["vapor_deficit"].head())

# 3. Ver cuántos nulos (NaN) reales hay tras hacer el resample mensual
monthly = df["vapor_deficit"].resample("ME").mean()
print(f"Meses totales: {len(monthly)}, Meses con NaN: {monthly.isna().sum()}")

# 4. Mostrar los meses que son NaN para localizarlos
if monthly.isna().sum() > 0:
    print("\nMeses con valores NaN:")
    print(monthly[monthly.isna()])

In [ ]:
fig, ax  = plt.subplots(figsize=(10,5), dpi=200)

monthly = df["vapor_deficit"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="dodgerblue", label = "Vapor Deficit") # i established that the date is the index
plt.ylabel("hPa")
avg = monthly.mean() 
plt.axhline(avg, color = "slateblue", linestyle = "--", label = f"Average Vapor Deficit: {avg:.1f} hPa")
plt.legend()
plt.xlabel("Year")

plt.title("Vapor Deficit (Tensión de vapor) 2012-2023")

ax.xaxis.set_major_locator(mdates.YearLocator(1))
plt.xticks(rotation = 45)

plt.grid()
plt.show()

In [ ]:
fig, ax  = plt.subplots(figsize=(10,5), dpi=200)

monthly = df["dew_point"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="skyblue", label = "Dew Point") # i established that the date is the index
mean = monthly.values.mean()
plt.axhline(mean, color = "red", label = f"Average Dew Point:{mean:.1f} ", linestyle = "--")
plt.ylabel("ºC")
plt.xlabel("Year")
plt.title("Dew Point 2012-2003")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,5), dpi=200)

anual = df["rain_mm"].resample("YE").sum()

ax.plot(anual.index, anual.values, marker = "o", linestyle = "-", color = "navy", label = "Anual Rainfall")
average = anual.mean()
ax.axhline(average, color = "red", linestyle = "--", label = f"Average ({average:.1f} mm)")

ax.set_ylabel("mm")
ax.set_xlabel("year")
ax.set_title("Anual Rainfall (2000-2023)")
ax.grid()
ax.legend()
plt.tight_layout()
plt.show()

#  Season graphs

In [ ]:
monthly = df.groupby(df.index.month)["temp"].agg(mean = "mean", p10 = lambda x: x.quantile(0.10), p90 = lambda x:  x.quantile(0.9))
print(monthly)

x = monthly.index

plt.figure(figsize = (10,5), dpi = 200)

plt.plot(x, monthly["mean"], color = "red", marker = "o", label = "Avg. Temperature")

plt.fill_between(
    x,
    monthly["p10"],
    monthly["p90"],
    alpha=0.2,
    color = "blue",
    label="10th–90th percentile"
)


plt.xticks(range(1,13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.xlim(1,12)
plt.grid()
plt.legend()
plt.title("Anual temperature cycle (2001-2023)")
plt.ylabel("Temperature ºC")
plt.tight_layout()
plt.show()

In [ ]:
temp = df.groupby(df.index.month)["temp"].mean()
print(temp)


rain = df["rain_mm"].resample("ME").sum().groupby(lambda d: d.month).mean() # lambda function does something like for d in indice clave = d.month

print(rain)

months = list(range(1,13))
labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
print(months)

fig, ax1 = plt.subplots(figsize = (10,5), dpi = 200)

ax1.plot(months, temp, color = "darkred", marker = "o",linewidth = 2.5)
ax2 = ax1.twinx()

ax2.bar(months, rain, color = "royalblue")

ax2.set_ylabel("Rain (mm)", color="royalblue")
ax1.set_ylabel("Tempertaure (ºC)", color = "darkred")
ax2.tick_params(axis="y", labelcolor="royalblue")
ax1.tick_params(axis = "y", labelcolor = "darkred")

ax1.set_xticks(months,labels)
ax1.set_xlim(0.5, 12.5)
ax1.set_zorder(2)
ax2.set_zorder(1)


plt.title("Salamanca's Cimate 2000-2023")
plt.tight_layout()

ax1.grid(axis = "y")
plt.show()